In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score


from src.utils import *
import src.prompt as prompt
from src.data_loader import load_spatial_data_onefile

In [ ]:
config = load_config("configs/config_zeroshot_copd.yaml")
config.data_name = "230267_Slide2"
config.replicate = "_uniquetest"
config.refresh_paths()
name_truth = config.name_truth



In [ ]:

niche_name = "kmeans_clusters"


# data

In [ ]:
data_path = str(dataset_dir("copd", "230267_Slide2"))
adata = load_spatial_data_onefile(f"{data_path}/sample_{config.data_name}_k20.csv", 
                                  config, 
                                  index_col=0)

In [ ]:
# Prepare neighbor data using the new function
neighbor_normalized_df, neighbor_normalized_df_genes, adj_matrix = prepare_neighbor_data(
    adata, config
)

# extra: rename _i to _inflamed

In [ ]:
neighbor_normalized_df.columns = neighbor_normalized_df.columns.str.replace("_i", "_inflamed")

# prompt


In [ ]:
domain_mapping = {0: "Airway Healthy",
                  1: "Airway Inflamed",
                  2: "Large Vessel Healthy",
                  3: "Large Vessel Inflamed",
                  4: "Alveolar",
                  5: "Fibrotic",
                  6: "Immune",
                  7: "Repair"}

config.domain_mapping = domain_mapping

In [ ]:

# IMPORTANT: check input and prompt_func
if config.Graph_type == "countPlusGenes":
    input_df = neighbor_normalized_df
    df_extra = neighbor_normalized_df_genes
    prompt_func = prompt.zeroshot_celltype_geneorder
elif config.Graph_type == "count":
    input_df = neighbor_normalized_df
    df_extra = None
    prompt_func = prompt.zeroshot_celltype
elif config.Graph_type == "GeneOnly":
    input_df = neighbor_normalized_df_genes
    df_extra = None
    prompt_func = prompt.zeroshot_geneorder
else:
    raise ValueError(f"Graph_type {config.Graph_type} not supported")

In [ ]:
print(prompt_func(input_df, [1], config))

# shorten redundent computation

In [ ]:
unique_indices, idx_mapping, inverse_mapping = get_unique_prompts(input_df, 
                                                    config, 
                                                    prompt_func, 
                                                    df_extra=df_extra)

input_df = input_df.iloc[unique_indices]
if df_extra is not None:
    df_extra = df_extra.iloc[unique_indices]

# gpt

## generate json

In [ ]:
# choose correct data and prompt
generate_json_end2end(input_df, 
                      config, 
                      prompt_func, 
                      batch_size = 5000,
                      max_completion_tokens = 512,  # key to control the cost, expecially for o3-mini
                      n_rows = 1,
                      df_extra = df_extra)

## submit

In [ ]:
# submit_end2end.py
# nohup python -u -m src.submit_end2end configs/config_BZ9_zeroshot.yaml > BZ9_zeroshot.out 2>&1 &
import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_zeroshot_copd.yaml {config.data_name} {config.replicate} > outs/{config.data_name}_{config.replicate}_zeroshot.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 2
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']

                # extract outputs - handle both JSON format and text format
                extract_dict = extract_json_microenvironment(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['zeroshot_gpt4o_mini']


In [ ]:
# # x = gpt_results_df.zeroshot_gpt4o_mini.loc[neighbor_normalized_df.index].values
# x = gpt_results_df.zeroshot_gpt4o_mini.loc[unique_df.index].values

### restore the full labels if use unique_df

In [ ]:
restored_full_df=restore_full_data_from_whole_df(neighbor_normalized_df, 
                                                gpt_results_df, 
                                                config, prompt_func, 
                                                df_extra=neighbor_normalized_df_genes)



In [ ]:
gpt_results_df = restored_full_df

In [ ]:
# # Calculate accuracy between truth (y) and predicted (x) values
# # First, make sure both arrays have the same length
# assert len(y) == len(x), "Arrays must have the same length to calculate accuracy"

# # Calculate the number of correct predictions
# correct_predictions = sum(y_val == x_val for y_val, x_val in zip(y, x))

# # Calculate accuracy as the proportion of correct predictions
# accuracy = correct_predictions / len(y)

# print(f"Accuracy: {accuracy:.4f} ({correct_predictions}/{len(y)} correct predictions)")


In [ ]:
gpt_results_df.zeroshot_gpt4o_mini.value_counts()

In [ ]:
# gpt_results_df.loc[gpt_results_df.zeroshot_gpt4o_mini == "Inflamed Airway"] = "Airway Inflammed"
gpt_results_df.loc[gpt_results_df.zeroshot_gpt4o_mini == "‘Airway Inflamed’"] = "Airway Inflamed"


In [ ]:
for nichtype in gpt_results_df.zeroshot_gpt4o_mini.value_counts()[gpt_results_df.zeroshot_gpt4o_mini.value_counts()<4].index:
    gpt_results_df.loc[gpt_results_df.zeroshot_gpt4o_mini == nichtype, "zeroshot_gpt4o_mini"] = "unknown"


In [ ]:
# gpt_results_df.columns = ['zeroshot_gpt4o']

# plot 

In [ ]:
adata.obs = adata.obs.join(gpt_results_df)
adata.obs['zeroshot_gpt4o_mini'] = adata.obs['zeroshot_gpt4o_mini'].fillna("unknown")


## refine

In [ ]:
adj_matrix, _ = sparse_adjacency(pos_data, threshold=config.r)
adata.obs['zeroshot_gpt4o_mini_refined'] = relabel_cells(adj_matrix.toarray(), adata.obs['zeroshot_gpt4o_mini'])


In [ ]:
# load previous results
results_df = pd.read_csv(f"{data_path}/{config.data_name}_zeroshot_gpt4o_refined.csv", index_col=0)
results_df = results_df.loc[adata.obs.index]
adata.obs['zeroshot_gpt4o_mini_refined'] = results_df['zeroshot_gpt4o_mini_refined']


In [ ]:
sc.pl.scatter(adata, x="x", y="y", color="zeroshot_gpt4o_mini", title =  f"zeroshot_gpt4o_mini")


In [ ]:
# adata.obs.to_csv(f"{data_path}/{config.data_name}_zeroshot_gpt4o_refined.csv")
# adata.obs.to_csv(f"{data_path}/{config.data_name}_zeroshot_gpt4o_refined_k20.csv")

In [ ]:
# Convert the niche column to categorical data type
# This helps with visualization and ensures consistent color mapping
niche_name = "k.niches.8"
adata.obs[niche_name] = adata.obs[niche_name].astype('category')



In [ ]:
sc.pl.scatter(adata, x="x", y="y", color=niche_name, title =  niche_name)


In [ ]:
adj_matrix, _ = sparse_adjacency(pos_data, threshold=config.r)

In [ ]:

# Calculate neighborhood ARI scores
neighborhood_ari = calculate_neighborhood_ari(adata.obs, 
                                              adj_matrix, 
                                              name_1=niche_name, 
                                              name_2='zeroshot_gpt4o_mini_refined')


In [ ]:

# Add the results back to adata.obs
adata.obs['neighborhood_ari'] = -neighborhood_ari['neighborhood_ari']

sc.pl.scatter(adata, x="x", y="y", color="neighborhood_ari", title =  f"neighborhood_ari")

In [ ]:
sc.pl.scatter(adata, x="x", y="y", color=celltype_key, title =  f"cell type")

In [ ]:
# Create a scatter plot of the data with neighborhood_ari as color
import matplotlib.pyplot as plt

# Set up the figure and axis
fig, ax = plt.subplots(figsize=(5, 4))

# Create scatter plot
scatter = ax.scatter(
    adata.obs['x'], 
    adata.obs['y'],
    c=adata.obs['neighborhood_ari'],
    cmap='viridis',
    s=1,  # Point size
    alpha=0.8  # Transparency
)

# Rescale color range from min to 0 (since we negated the values earlier)
min_val = adata.obs['neighborhood_ari'].min()
scatter.set_clim(min_val, 0)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Neighborhood ARI')

# Set labels and title
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Neighborhood ARI Visualization')

# Remove ticks for cleaner look
ax.set_xticks([])
ax.set_yticks([])

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()


# Gemini


In [ ]:
import google.generativeai as genai
import pickle
import time



genai.configure(api_key=os.environ["API_KEY"])
model = genai.GenerativeModel("gemini-1.5-pro")
gen_config=genai.types.GenerationConfig(temperature=1.0, max_output_tokens=1000)



In [ ]:
gemini_results_df, store_responses = run_gemini(model, gen_config, 
                                                neighbor_normalized_df, config, 
                                                prompt.zeroshot_celltype, n_rows=1, 
                                                column_name="zeroshot_gemini")

In [ ]:
gemini_results_df = pd.read_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv", index_col=0)

In [ ]:
gemini_results_df.columns = ['zeroshot_gemini']


In [ ]:
gemini_results_df.index = adata.obs.index

In [ ]:
adata.obs = adata.obs.join(gemini_results_df)
adata.obs['zeroshot_gemini'] = adata.obs['zeroshot_gemini'].fillna("unknown")
adj_matrix, _ = sparse_adjacency(pos_data, threshold=config.r)
adata.obs['zeroshot_gemini_refined'] = relabel_cells(adj_matrix.toarray(), adata.obs['zeroshot_gemini'])


In [ ]:
sc.pl.scatter(adata, x="x", y="y", color="zeroshot_gemini_refined", title =  f"zeroshot_gemini_refined")

In [ ]:
adata.obs.to_csv(f"{data_path}/{config.data_name}_zeroshot_gemini_refined.csv")